# TE Transfer Visualizations

This notebook creates before-and-after charts for tight ends transferring from non-power to power conferences. First, run the data export cell in `te_transfer_analysis.ipynb`; this notebook reads the resulting `te_np_to_p.csv` file.

In [6]:
# Before vs After Transfer

from pathlib import Path
from IPython.display import HTML, display
import pandas as pd
import plotly.express as px

# Support running from either the workspace root or this notebook's folder.
candidate_paths = [
    Path("te_np_to_p.csv"),
    Path("code/Positions/TE/te_np_to_p.csv"),
]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "te_np_to_p.csv was not found. Run the export cell in "
        "te_transfer_analysis.ipynb first."
    )

te_np_to_p = pd.read_csv(data_path)

metric_labels = {
    "receiving_REC": "Receptions",
    "receiving_YDS": "Receiving Yards",
    "receiving_YPR": "Yards per Reception",
    "receiving_TD": "Receiving Touchdowns",
    "adjPPA": "Adjusted PPA",
}

before_after_rows = []
for metric, label in metric_labels.items():
    pre_col = f"{metric}_pre"
    post_col = f"{metric}_post"
    paired_values = (
        te_np_to_p[[pre_col, post_col]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
    )

    if paired_values.empty:
        continue

    before_after_rows.extend([
        {"Metric": label, "Period": "Before", "Average": paired_values[pre_col].mean()},
        {"Metric": label, "Period": "After", "Average": paired_values[post_col].mean()},
    ])

before_after_summary = pd.DataFrame(before_after_rows)

fig = px.bar(
    before_after_summary,
    x="Average",
    y="Period",
    text="Average",
    color="Period",
    facet_col="Metric",
    facet_col_wrap=1,
    title="Tight End Performance Before vs. After Non-Power → Power Transfer",
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.D3,
    category_orders={
        "Period": ["Before", "After"],
        "Metric": list(metric_labels.values()),
    },
    orientation="h",
)
fig.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.split("=")[-1],
        x=0,
        xanchor="left",
    )
)
fig.update_traces(
    texttemplate="%{x:,.2f}",
    textposition="outside",
    cliponaxis=False,
)
fig.update_layout(
    showlegend=False,
    xaxis_title="Average (each metric has its own scale)",
    yaxis_title="",
    height=1000,
    title_x=0.5,
    margin=dict(l=110),
)
fig.update_xaxes(matches=None, rangemode="tozero")
fig.update_yaxes(categoryorder="array", categoryarray=["Before", "After"], autorange="reversed", title_text="")

display(HTML(fig.to_html(include_plotlyjs="cdn")))
before_after_summary

,Metric,Period,Average
0,Receptions,Before,23.387755
1,Receptions,After,15.591837
2,Receiving Yards,Before,255.346939
3,Receiving Yards,After,168.857143
4,Yards per Reception,Before,10.736735
5,Yards per Reception,After,9.900000
6,Receiving Touchdowns,Before,2.408163
7,Receiving Touchdowns,After,1.428571
8,Adjusted PPA,Before,0.712858
9,Adjusted PPA,After,0.660799


In [4]:
# Distribution of receiving-yard change
yard_change = (
    pd.to_numeric(te_np_to_p["receiving_YDS_post"], errors="coerce")
    - pd.to_numeric(te_np_to_p["receiving_YDS_pre"], errors="coerce")
).dropna()

yard_change_df = pd.DataFrame({"Receiving Yard Change": yard_change})
fig_yard_change = px.histogram(
    data_frame=yard_change_df,
    x="Receiving Yard Change",
    nbins=20,
    title=f"Distribution of Receiving-Yard Change After NP → P Transfer (n={len(yard_change)})",
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.D3,
)
fig_yard_change.add_vline(x=0, line_dash="dash", line_color="red")
fig_yard_change.update_layout(
    xaxis_title="Change in Receiving Yards (post-transfer minus pre-transfer)",
    yaxis_title="Number of Tight Ends",
    title_x=0.5,
)
display(HTML(fig_yard_change.to_html(include_plotlyjs="cdn")))

In [8]:
# Percent of tight ends who improved in each metric
improvement_rows = []

for metric, label in metric_labels.items():
    pre_col = f"{metric}_pre"
    post_col = f"{metric}_post"
    paired_values = (
        te_np_to_p[[pre_col, post_col]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
    )

    if paired_values.empty:
        continue

    percent_improved = (paired_values[post_col] > paired_values[pre_col]).mean() * 100
    improvement_rows.append({"Metric": label, "Percent Improved": percent_improved})

improvement_summary = (
    pd.DataFrame(improvement_rows)
    .sort_values("Percent Improved", ascending=False)
)

fig_improvement = px.bar(
    improvement_summary,
    x="Percent Improved",
    y="Metric",
    text="Percent Improved",
    title="Percent of Tight Ends Who Improved After NP → P Transfer",
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.D3,
    orientation="h",
    category_orders={"Metric": improvement_summary["Metric"].tolist()},
)
fig_improvement.update_traces(
    texttemplate="%{x:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_improvement.update_layout(
    xaxis_title="Players with a higher post-transfer value (%)",
    yaxis_title="",
    title_x=0.5,
    margin=dict(l=180),
)
fig_improvement.update_xaxes(range=[0, 100], ticksuffix="%")
fig_improvement.update_yaxes(autorange="reversed")

display(HTML(fig_improvement.to_html(include_plotlyjs="cdn")))
improvement_summary

,Metric,Percent Improved
2,Yards per Reception,44.897959
4,Adjusted PPA,40.000000
3,Receiving Touchdowns,24.489796
0,Receptions,18.367347
1,Receiving Yards,16.326531
